# Обучение YOLOv8 на своём объекте и конвертация в RKNN для Luckfox Pico Mini

Этот ноутбук делает всё, что нужно между "у меня есть размеченный датасет" и
"у меня есть файл yolov8.rknn, готовый для платы":

1. Обучение YOLOv8 (Ultralytics) на вашем датасете
2. Экспорт обученной модели в ONNX специальным форком от Rockchip (без этого RKNN
   работать откажется - убираются несовместимые с NPU операции)
3. Квантизация в INT8 и конвертация в RKNN (rknn-toolkit2 + rknn_model_zoo)

Перед началом: **Среда выполнения → Сменить среду выполнения → GPU (T4)** — обучение
на CPU займёт значительно больше времени.

Датасет должен быть в формате YOLOv8 (см. docs/04-dataset.md в репозитории):
zip-архив, внутри train/valid (images/ + labels/) и data.yaml.

## 1. Установка зависимостей

In [ ]:
!pip install ultralytics -U -q


## 2. Загрузка датасета

Вариант А — через Google Drive (удобно, если планируете возвращаться к ноутбуку):

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Поправьте путь к вашему zip-архиву с датасетом и распакуйте его:

In [ ]:
import os

DATASET_ZIP = "/content/drive/MyDrive/my_dataset.zip"  # <-- поменяйте на свой путь
DATASET_DIR = "/content/my_dataset"

%cd /content
!unzip -o "{DATASET_ZIP}" -d "{DATASET_DIR}"


Вариант Б — просто загрузить zip прямо в сессию (иконка "Файлы" на панели слева →
"Загрузить в сессионное хранилище"), тогда `DATASET_ZIP` укажите как `/content/имя_архива.zip`.

## 3. Проверяем и правим data.yaml

Ultralytics надёжнее всего работает с абсолютными путями. Откройте
`{DATASET_DIR}/data.yaml` в файловом менеджере слева и убедитесь, что пути
`train:` / `val:` абсолютные, например:

```yaml
train: /content/my_dataset/train/images
val: /content/my_dataset/valid/images
nc: 1
names: ['my_object']
```

## 4. Обучение

In [ ]:
import os
os.environ['WANDB_MODE'] = 'disabled'  # чтобы не спрашивал логин в Weights & Biases

from ultralytics import YOLO

IMG_SIZE = 640          # тот же размер, что и в статье; можно 320 для большего FPS на плате
EPOCHS = 100
BATCH = 16              # уменьшите, если не хватает видеопамяти

model = YOLO('yolov8n.pt')
results = model.train(
    data=f"{DATASET_DIR}/data.yaml",
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=0,
)


## 5. Проверка качества (mAP)

In [ ]:
metrics = model.val(split='val', plots=True)


## 6. Сохраняем веса на Google Drive (на всякий случай)

In [ ]:
!cp -r /content/runs/detect/train /content/drive/MyDrive/yolo_train_backup
print("Веса тут: /content/runs/detect/train/weights/best.pt")


## 7. Экспорт в ONNX (форк Rockchip)

Обычный `model.export(format='onnx')` от Ultralytics **не подойдёт** - RKNN не
поддерживает часть операций (постобработка, DFL-блок). Нужен специальный форк,
который убирает эти узлы из графа перед экспортом.

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/airockchip/ultralytics_yolov8
%cd ultralytics_yolov8
!pip install onnx onnxruntime -q


Теперь нужно вручную открыть файл
`/content/ultralytics_yolov8/ultralytics/cfg/default.yaml` (через файловый менеджер
слева) и поменять в нём:

- `model:` → путь к вашим весам, например `/content/runs/detect/train/weights/best.pt`
- `imgsz:` → тот же размер, что при обучении (по умолчанию в файле 640 — если обучали
  на 640, можно не трогать)

После этого запускаем экспорт:

In [ ]:
!PYTHONPATH=./ python ./ultralytics/engine/exporter.py


In [ ]:
ONNX_PATH = "/content/runs/detect/train/weights/best.onnx"
!ls -la "{ONNX_PATH}"
!cp "{ONNX_PATH}" /content/drive/MyDrive/my_model.onnx


## 8. Конвертация в RKNN (квантизация INT8)

NPU процессора RV1103 не умеет считать во float, модель нужно квантизовать в int8.
Для этого нужна калибровка на части изображений вашего датасета (чем больше — тем
точнее, но дольше; 20-50 изображений обычно достаточно).

In [ ]:
%cd /content
!pip install -q --no-deps \
    https://raw.githubusercontent.com/airockchip/rknn-toolkit2/refs/heads/master/rknn-toolkit2/packages/rknn_toolkit2-2.2.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
import rknn
print("rknn-toolkit2 установлен")


In [ ]:
!git clone --depth 1 https://github.com/airockchip/rknn_model_zoo
%cd /content/rknn_model_zoo/examples/yolov8


Формируем список файлов для калибровки (можно взять все train-изображения, либо
срез `files[:40]`, если датасет большой и калибровка идёт слишком долго):

In [ ]:
import os

IMG_PATH = f"{DATASET_DIR}/train/images"
files = [f for f in os.listdir(IMG_PATH) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
print("изображений для калибровки:", len(files))

with open("/content/data_subset.txt", "w") as fd:
    for name in files:
        print(os.path.join(IMG_PATH, name), file=fd)


Откройте `/content/rknn_model_zoo/examples/yolov8/python/convert.py` и убедитесь,
что путь к списку калибровки указывает на `/content/data_subset.txt` (обычно это
переменная `DATASET_PATH` в начале файла).

In [ ]:
%cd python
!python3 convert.py "{ONNX_PATH}" rv1103 i8


Если всё прошло без ошибок, результат появится в `../model/yolov8.rknn`. Скачиваем
и сохраняем — этот файл нужен на следующем шаге (docs/06-build-and-deploy.md).

In [ ]:
RKNN_OUT = "/content/rknn_model_zoo/examples/yolov8/model/yolov8.rknn"
!cp "{RKNN_OUT}" /content/drive/MyDrive/my_model.rknn

from google.colab import files
files.download(RKNN_OUT)


## 9. labels.txt

Создайте (если ещё не создан) текстовый файл `labels.txt` — по одному имени класса
на строку, в том же порядке, что в `names` из вашего `data.yaml`. Он нужен на плате
рядом с моделью, а также при сборке `HelloDetector` (шаг 06).

In [ ]:
names = ['my_object']  # <-- поставьте те же имена и в том же порядке, что в data.yaml
with open('/content/labels.txt', 'w') as fd:
    fd.write('\n'.join(names) + '\n')

from google.colab import files
files.download('/content/labels.txt')
